# 🚨 تحليل كود التنبيهات (Alerts) — Sprint 4

---

## 🎯 TL;DR

> **3 كائنات أساسية**: `AlertStatus` (حالة)، `ConditionOperator` (معامل مقارنة)، `AlertCondition` (شرط مثل `rsi < 30`). ثم **`Alert`** الذي يجمعها مع `pair` و`interval`. **الهدف**: مراقبة تلقائية وإطلاق تنبيه عند تحقق شرط.

---

## 📊 المخطط

```mermaid
graph TD
    A[Alert] --> B[AlertCondition]
    A --> C[AlertStatus]
    B --> E[ConditionOperator]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style B fill:#bbf,stroke:#333,stroke-width:2px
```

---

## 🔬 الكائنات

### 1️⃣ `AlertStatus` — الحالات

`ACTIVE` (نشط) → `TRIGGERED` (تم الإطلاق) → `DISABLED` (معطّل).

**🎯 ترث من `str, Enum`** → قابلة للمقارنة كنص + تخزين في DB.

---

### 2️⃣ `ConditionOperator` — المقارنات

`>`, `>=`, `<`, `<=`, `==` مع `from_str()` للتحويل.

---

### 3️⃣ `AlertCondition` — الشرط (Value Object `frozen`)

```python
metric: str           # 'price', 'rsi', 'sma'
operator: ConditionOperator
target_value: Decimal
```

#### `parse("rsi <= 30")`

```mermaid
graph TD
    A["'rsi <= 30'"] --> B[find operator]
    B --> C["split by op"]
    C --> D["metric='rsi', value='30'"]
    D --> E[AlertCondition]

    style B fill:#bbf,stroke:#333
    style E fill:#bfb,stroke:#333
```

**🎯 تريكة ترتيب المعاملات:** `[">=", "<=", "==", ">", "<"]` — **الأطول أولاً** لتفادي الخلط (`>=` قبل `>`).

**🎯 `split(op, 1)`**: maxsplit=1 → تقسيمان فقط.

#### `evaluate(current)` — 5 مقارنات

**🎯 `None → False`**: Fail Soft — لا تنبيه بدون بيانات.

---

### 4️⃣ `Alert` — الكائن الرئيسي (غير frozen)

| الحقل | Default | الدور |
|-------|---------|-------|
| `id` | `uuid4()[:8]` | معرّف فريد |
| `interval` | `"1day"` | الإطار |
| `status` | `ACTIVE` | يبدأ نشطاً |
| `message` | auto في `__post_init__` | وصف |
| `created_at` | `datetime.now` | وقت الإنشاء |
| `triggered_at` | `None` | عند الإطلاق |

**🎯 لماذا غير frozen؟** لأن `status` و`triggered_at` **يتغيران** عند `trigger()`.

#### `trigger()` — الإطلاق

`status = TRIGGERED` + `triggered_at = now`.

#### `to_dict()` / `from_dict()`

**🎯 التحدي:** `Decimal` + `datetime` + `Enum` + كائن متداخل → يُسطَّح في `to_dict`.

**🎯 `str(Decimal)`**: غير JSON-serializable مباشرة.

---

## 📊 دورة حياة `Alert`

```mermaid
stateDiagram-v2
    [*] --> ACTIVE: إنشاء
    ACTIVE --> TRIGGERED: trigger()
    ACTIVE --> DISABLED: تعطيل
    DISABLED --> ACTIVE: تفعيل

    note right of TRIGGERED
        triggered_at يُملأ
    end note
```

---

## 🧪 اختبار (وصف)

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | `parse("rsi <= 30")` | metric=rsi, op=LE, value=30 |
| 2 | `parse("invalid")` | `ValidationError` |
| 3 | `evaluate(None)` | `False` |
| 4 | `Alert(...)` | id + message تلقائي + ACTIVE |
| 5 | `trigger(...)` | status=TRIGGERED |
| 6 | `to_dict` → `from_dict` | مطابق |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `metric: str` بدون تحقق | يقبل `"invalid"` |
| لا `!=` في `ConditionOperator` | غير حرج |
| `from_dict` بـ `data["metric"]` | `KeyError` لو ناقص |
| `trigger` لا يتحقق من `ACTIVE` | يمكن إطلاق معطّل |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| Immutability | `AlertCondition` frozen |
| Mutable State | `Alert` غير frozen |
| Fail Soft | `None → False` |
| Fail Early | `parse` → `ValidationError` |
| Serialization | `to_dict`/`from_dict` |
| Domain Enum | `AlertStatus`, `ConditionOperator` |

---

## 🗺️ Sprint 4

```mermaid
graph LR
    A["core/alert.py ✅"] --> B["AlertRepository"]
    B --> C["AlertEngine"]
    C --> D["CLI: alert add/list"]
    D --> E["tests"]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style B fill:#ddd,stroke:#333
```

---

## 📋 الخطوة التالية

> **`AlertRepository`** — تخزين في SQLite:
> `save`, `list_active`, `get_by_id`, `update_status`.

**السؤال الجوهري:** كيف نفحص التنبيهات دورياً؟

**الجواب:** `AlertEngine.check_all()` → يجلب القيمة → `evaluate` → `trigger` لو تحقق. عبر CLI أو Cron.

In [1]:
from decimal import Decimal
from core.alert import AlertCondition

# 1. اختبار تحليل الشرط
cond = AlertCondition.parse('price >= 1.0850')
print('Parsed:', cond.metric, cond.operator.value, cond.target_value)

# 2. اختبار التقييم
print('Eval 1.0860:', cond.evaluate(Decimal('1.0860')))  # True
print('Eval 1.0840:', cond.evaluate(Decimal('1.0840')))  # False

Parsed: price >= 1.0850
Eval 1.0860: True
Eval 1.0840: False


# 💾 تحليل `AlertRepository` — تخزين التنبيهات في SQLite (Sprint 4، الخطوة 2)

---

## 🎯 TL;DR

> **`Storage` ABC** يحصل على 4 دوال جديدة (`save_alert`, `get_alerts`, `get_alert_by_id`, `delete_alert`). **`SQLiteStorage`** ينشئ جدول `alerts` + فهرساً، ويُنفّذ الـ CRUD. **`MemoryStorage`** يحاكيها بـ `dict` للاختبارات.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[Storage ABC] --> B[SQLiteStorage]
    A --> C[MemoryStorage]
    B --> D[alerts table]
    C --> E[dict _alerts]
    
    F[Alert.to_dict] --> B
    B --> G[Alert.from_dict]

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style B fill:#bfb,stroke:#333,stroke-width:2px
    style C fill:#f9f,stroke:#333
```

---

## 🔬 التشريح — 4 أجزاء

### 1️⃣ العقد (`storage/base.py`) — 4 abstract methods

| الدالة | التوقيع | الدور |
|--------|---------|-------|
| `save_alert` | `(alert) -> None` | Insert أو Replace |
| `get_alerts` | `(active_only=False) -> list` | الكل أو النشط فقط |
| `get_alert_by_id` | `(alert_id) -> Alert \| None` | واحد بالمعرّف |
| `delete_alert` | `(alert_id) -> bool` | حذف + إرجاع هل تم |

**🎯 لماذا `Any` بدل `Alert`؟** لتجنب **Circular Import** — `storage/base.py` قد يُستورد من `core/`، و`Alert` في `core/`. في الكود الفعلي يُستورد `Alert` بأمان.

---

### 2️⃣ جدول `alerts` في `SQLiteStorage`

```mermaid
erDiagram
    ALERTS {
        TEXT id PK
        TEXT pair
        TEXT interval
        TEXT metric
        TEXT operator
        TEXT target_value
        TEXT status
        TEXT message
        TEXT created_at
        TEXT triggered_at
    }
```

**🎯 القرارات:**

| الحقل | النوع | السبب |
|-------|-------|-------|
| `id` | `TEXT PK` | UUID قصير — لا حاجة لـ autoincrement |
| `target_value` | `TEXT` | يحفظ دقة `Decimal` |
| `status` | `TEXT` | enum value — قابل للفلترة |
| `created_at` | `TEXT` | ISO format |
| `triggered_at` | `TEXT NULL` | يُملأ عند الإطلاق فقط |

**🎯 الفهرس:** `idx_alerts_status_pair` على `(status, pair)` — يُسرّع `active_only=True` (الاستعلام الأكثر استخداماً).

---

### 3️⃣ تريكة الترحيل التلقائي (Auto-Migration)

```python
cursor.execute("PRAGMA table_info(candles)")
existing_columns = [row["name"] for row in cursor.fetchall()]
if "interval" not in existing_columns:
    cursor.execute("ALTER TABLE candles ADD COLUMN interval TEXT NOT NULL DEFAULT '1day'")
```

**🎯 المشكلة:** قواعد بيانات Sprint 1 **ليس فيها** عمود `interval`. لو شغّلت النسخة الجديدة على DB قديمة → `OperationalError: no such column`.

**🎯 الحل:** فحص `PRAGMA table_info` قبل `ALTER TABLE` — إضافة **آمنة وذاتية**.

| السيناريو | النتيجة |
|-----------|---------|
| DB جديدة | `CREATE TABLE` ينشئ كل الأعمدة |
| DB قديمة (بدون interval) | `ALTER TABLE` يضيف العمود |
| DB حديثة | لا يفعل شيئاً ✅ |

---

### 4️⃣ `MemoryStorage` — محاكاة بـ dict

| الدالة | التنفيذ |
|--------|---------|
| `save_alert` | `self._alerts[alert.id] = alert` |
| `get_alerts` | `list(values)` + filter + sort |
| `get_alert_by_id` | `self._alerts.get(id)` |
| `delete_alert` | `self._alerts.pop(id, None) is not None` |

**🎯 تريكة `pop(id, None) is not None`:** ترجع `True` لو المفتاح كان موجوداً، `False` لو لا. **بدون KeyError**.

**🎯 تريكة `hasattr(self, "_alerts")`:** حماية لو لم تُهيَّأ في `__init__` — **Defensive**.

---

## 📊 دورة حفظ/استرجاع

```mermaid
sequenceDiagram
    participant S as Service
    participant St as Storage
    participant DB as SQLite

    S->>St: save_alert(alert)
    St->>St: alert.to_dict()
    St->>DB: INSERT OR REPLACE (named)
    DB-->>St: OK
    St->>DB: commit
    St-->>S: None

    S->>St: get_alerts(active_only=True)
    St->>DB: SELECT WHERE status='ACTIVE'
    DB-->>St: rows
    St->>St: Alert.from_dict(dict(row))
    St-->>S: list[Alert]
```

---

## 🧪 التحقق

| # | الاختبار | المتوقع |
|---|----------|---------|
| 1 | `save_alert` + `get_alerts(active_only=True)` | 1 تنبيه |
| 2 | `get_alert_by_id` بعد الحفظ | نفس التنبيه |
| 3 | `delete_alert` | `True` + لا يظهر في `get_alerts` |
| 4 | `delete_alert` على ID غير موجود | `False` |
| 5 | `get_alerts` على DB فارغة | `[]` |
| 6 | DB قديمة بدون `interval` | `ALTER TABLE` ينجح |
| 7 | `:memory:` | معزول بين الاختبارات |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `save_alert` بـ `INSERT OR REPLACE` | يفقد `created_at` الأصلي لو تحديث؟ **لا** — `to_dict` يحمل القيمة الحالية |
| `get_alerts` بدون `limit` | يمكن أن يُرجع آلاف الصفوف |
| `delete_alert` لا يتحقق من وجود | `rowcount > 0` كافٍ |
| `_alerts` كـ dict في Memory | `save` ثانٍ لنفس الـ id يستبدل |
| `active_only` يستخدم `"ACTIVE"` نصياً | لو Enum تغير → كسر |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| Contract Enforcement | 4 abstract methods في ABC |
| Auto-Migration | `PRAGMA table_info` + `ALTER TABLE` |
| Indexing | `idx_alerts_status_pair` |
| DRY | `save_alert` يستخدم `to_dict` |
| Fail Safe | `pop(id, None)` بدون KeyError |
| Decimal Precision | `target_value` كـ TEXT |
| DI | `MemoryStorage` للاختبارات |
| ISO Serialization | `created_at`/`triggered_at` |

---

## 🗺️ Sprint 4

```mermaid
graph LR
    A["core/alert.py ✅"] --> B["AlertRepository 🆕"]
    B --> C["AlertService + Engine"]
    C --> D["CLI: alert add/list/check"]
    D --> E["tests/test_alerts.py"]

    style B fill:#bfb,stroke:#333,stroke-width:3px
    style C fill:#ddd,stroke:#333
```

---

## 📋 الخطوة التالية

> **`services/alert_service.py`** — محرك التنبيهات:
> - `create_alert(pair, condition_str, interval)` → يبني `Alert` ويحفظه
> - `list_alerts(active_only)` → يقرأ من Storage
> - `check_all()` → يمر على النشطة، يجلب القيمة (price/RSI/SMA)، `evaluate`، `trigger` لو تحقق

**السؤال الجوهري:** كيف يحصل `AlertService` على القيمة الحالية للمقياس (`price`, `rsi`, `sma`)؟

**الجواب:** يستخدم `ForexService` (Sprint 3):
- `metric="price"` → `get_current_price(pair)`
- `metric="rsi"` → `analyze_pair(pair, "rsi")` → آخر قيمة
- `metric="sma"` → نفس الشيء

**التحدي:** التوحيد بين أسماء المقاييس — يحل بـ **metric registry** أو `if/elif` (YAGNI الآن).

In [2]:
from core.alert import Alert, AlertCondition
from storage.sqlite_storage import SQLiteStorage

storage = SQLiteStorage(db_path=':memory:')

# إنشاء وحفظ تنبيه
alert = Alert(pair='EUR/USD', condition=AlertCondition.parse('price > 1.0900'), message='Target reached')
storage.save_alert(alert)

# استرجاع التنبيهات
loaded = storage.get_alerts(active_only=True)
print(f'Saved & Retrieved Alert ID: {loaded[0].id} | Pair: {loaded[0].pair} | Condition: {loaded[0].condition.metric} {loaded[0].condition.operator.value} {loaded[0].condition.target_value}')


[2026-11-05 23:28:59] [INFO] [forex_tracker]: Connected to SQLite database at: :memory:
[2026-11-05 23:28:59] [INFO] [forex_tracker]: Alert d460d290 (EUR/USD) saved successfully.


Saved & Retrieved Alert ID: d460d290 | Pair: EUR/USD | Condition: price > 1.0900


In [3]:
from storage.sqlite_storage import SQLiteStorage

# الاتصال بقاعدة البيانات الفعلية على القرص وتشغيل _init_db
storage = SQLiteStorage()
print('Database initialized on disk at:', storage.db_path)

[2026-11-05 23:30:12] [INFO] [forex_tracker]: Connected to SQLite database at: ./data\candles.db


Database initialized on disk at: ./data\candles.db


# 🔔 تحليل `AlertService` — محرك التنبيهات (Sprint 4، الخطوة 3)

---

## 🎯 TL;DR

> **`AlertService`** يربط `Storage` (حفظ/قراءة) مع `ForexService` (بيانات/مؤشرات). يُدير **دورة حياة التنبيه** ويُشغّل **محرك الفحص** الذي يقيّم الشروط ويُطلق التنبيهات.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[AlertService] --> B[Storage]
    A --> C[ForexService]
    A --> D[Alert]
    
    E[create_alert] --> D
    E --> B
    F[check_alerts] --> B
    F --> G[_get_metric_value]
    G --> C
    F --> H[trigger + save]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style G fill:#bbf,stroke:#333,stroke-width:2px
    style H fill:#f9f,stroke:#333
```

---

## 🔬 التشريح — 6 دوال

### 1️⃣ `__init__` — DI مزدوج

`ForexService(storage=self.storage)` — **نفس الـ storage** للاثنين (مهم في `:memory:`).

---

### 2️⃣ `create_alert` — تحقق ثلاثي

```mermaid
graph LR
    A[pair] --> B[CurrencyPair]
    C[interval] --> D[Interval]
    E[condition] --> F[AlertCondition.parse]
    B --> G[Alert]
    D --> G
    F --> G
    G --> H[save_alert]

    style H fill:#bfb,stroke:#333,stroke-width:2px
```

**Fail Early**: توحيد `eurusd` → `EUR/USD` + `1d` → `1day` + صيغة الشرط.

---

### 3️⃣ `list_alerts` — Thin wrapper

`storage.get_alerts(active_only=...)`.

---

### 4️⃣ `delete_alert` — Fail Soft

`True`/`False` بدون exception لو غير موجود.

---

### 5️⃣ `check_alerts` — **قلب النظام**

```mermaid
graph TD
    A[get active alerts] --> B{pair filter?}
    B --> C[filter by symbol]
    C --> D[loop alerts]
    D --> E[_get_metric_value]
    E --> F{not None<br/>AND evaluate?}
    F -->|Yes| G[trigger]
    G --> H[save_alert]
    H --> I[append report]
    I --> D
    F -->|No| D
    D --> J[return reports]

    style F fill:#f9f,stroke:#333,stroke-width:2px
    style H fill:#bfb,stroke:#333,stroke-width:2px
```

**التقرير — 9 حقول:**

| الحقل | الدور |
|-------|-------|
| `alert_id`, `pair`, `interval` | السياق |
| `metric`, `operator`, `target_value` | الشرط |
| `actual_value` | القيمة الفعلية |
| `message`, `triggered_at` | الوصف والتوقيت |

**🎯 تريكة `save_alert` بعد `trigger`:** منع الإطلاق المتكرر — التنبيه يصبح `TRIGGERED` فوراً.

---

### 6️⃣ `_get_metric_value` — ذكاء الاستخراج

```mermaid
graph TD
    A[metric] --> B{Type?}
    B -->|price/close/open/high/low| C[get_history limit=1]
    B -->|rsi/sma/ema| D[analyze_pair limit=35]
    C --> E[extract field]
    D --> F[key_map → rsi_14/sma_20/ema_20]
    F --> G[results[-1].value]
    E --> H[return Decimal | None]

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style H fill:#bfb,stroke:#333
```

| الفئة | المقاييس | المصدر |
|-------|---------|--------|
| **سعرية** | `price`, `close`, `open`, `high`, `low` | `get_history(limit=1)` |
| **فنية** | `rsi`, `sma`, `ema` | `analyze_pair(limit=35)` |

**🎯 تريكة Aliases:** `"rsi"` و `"rsi_14"` → نفس المؤشر.

**🎯 تريكة `limit=35`:** يكفي MACD (34) — هامش أمان.

**🎯 تريكة `None` عند الفشل:** `evaluate` يعالجها بـ `False`.

---

## 🧪 التحقق — دورة كاملة

| # | الخطوة | المتوقع |
|---|--------|---------|
| 1 | `create_alert("eurusd", "price > 1.0920")` | Alert ACTIVE |
| 2 | `save_candle` بسعر `1.0950` | في DB |
| 3 | `check_alerts()` | 1 triggered |
| 4 | `get_alert_by_id` | TRIGGERED |
| 5 | `check_alerts()` مجدداً | 0 (لا تكرار) |

**🎯 جوهر الاختبار:** إثبات **Idempotency** — بعد الإطلاق، التنبيه ليس `ACTIVE`.

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `check_alerts` يحفظ واحداً واحداً | بطيء لو 100 تنبيه |
| `analyze_pair` لكل تنبيه | استهلاك DB/API متكرر |
| `if/elif` طويلة في `_get_metric_value` | يمكن استبدالها بـ registry |
| `limit=35` ثابت | لا يدعم RSI(50) |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| Orchestration | يربط Storage + ForexService |
| DI متسق | نفس `storage` للـ Service والـ Forex |
| Fail Early | تحقق ثلاثي قبل الحفظ |
| Fail Soft | `None` لا يرمي |
| Idempotency | `save_alert` بعد `trigger` يمنع التكرار |
| Metric Aliases | `rsi` = `rsi_14` |
| Structured Report | dict بـ 9 حقول |

---

## 🗺️ Sprint 4

```mermaid
graph LR
    A["core/alert.py ✅"] --> B["AlertRepository ✅"]
    B --> C["AlertService 🆕"]
    C --> D["CLI: alert commands"]
    D --> E["tests"]

    style C fill:#bfb,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **CLI**: `forex alert add/list/check/delete`

| الأمر | الوظيفة |
|-------|---------|
| `forex alert add --pair EUR/USD --condition "price > 1.09"` | إنشاء |
| `forex alert list --active` | عرض النشطة |
| `forex alert check` | فحص يدوي |
| `forex alert delete --id abc12345` | حذف |

**النمط:** نفس `cmd_list` — أعمدة ثابتة: `ID | Pair | Condition | Status | Created`.

In [ ]:
from decimal import Decimal
from core.alert import AlertStatus
from services.alert_service import AlertService
from services.forex_service import ForexService
from storage.sqlite_storage import SQLiteStorage

storage = SQLiteStorage(db_path=':memory:')
forex_service = ForexService(storage=storage)
alert_service = AlertService(storage=storage, forex_service=forex_service)

# 1. إنشاء شمعة سعرية (EUR/USD سعر الإغلاق 1.0950)
from datetime import datetime
from core.candle import Candle
candle = Candle(
    pair='EUR/USD', interval='1day', timestamp=datetime.now(),
    open=Decimal('1.0900'), high=Decimal('1.0960'), low=Decimal('1.0890'), close=Decimal('1.0950')
)
storage.save_candle(candle)

# 2. إنشاء تنبيه نشط: price > 1.0920
alert = alert_service.create_alert(pair='EUR/USD', condition_str='price > 1.0920', message='EUR Bullish Break')
print('Alert Created ID:', alert.id, '| Status:', alert.status.value)

# 3. فحص التنبيهات
triggered = alert_service.check_alerts()
print('Triggered Count:', len(triggered))
if triggered:
    t = triggered[0]
    print("Triggered Details: {} actual={} target={}".format(
        t["pair"], t["actual_value"], t["target_value"]
    ))

# 4. التأكد من تحول حالة التنبيه في قاعدة البيانات إلى TRIGGERED
reloaded = storage.get_alert_by_id(alert.id)
assert reloaded is not None
print('Updated DB Status:', reloaded.status.value)

[2026-11-05 23:42:53] [INFO] [forex_tracker]: Connected to SQLite database at: :memory:
[2026-11-05 23:42:53] [INFO] [forex_tracker]: Saved 1 candle(s) to SQLite.
[2026-11-05 23:42:53] [INFO] [forex_tracker]: Alert 1f142c86 (EUR/USD) saved successfully.
[2026-11-05 23:42:53] [INFO] [forex_tracker]: Created alert 1f142c86 for EUR/USD [1day]: price > 1.0920
[2026-11-05 23:42:53] [INFO] [forex_tracker]: Alert 1f142c86 (EUR/USD) saved successfully.
[2026-11-05 23:42:53] [WARNING] [forex_tracker]: 🔔 ALERT TRIGGERED [1f142c86]: EUR/USD price is 1.0950 (> 1.0920)


Alert Created ID: 1f142c86 | Status: ACTIVE
Triggered Count: 1
Triggered Details: EUR/USD actual=1.0950 target=1.0920
Updated DB Status: TRIGGERED


# 🖥️ تحليل أوامر التنبيهات في CLI (Sprint 4، الخطوة 4)

---

## 🎯 TL;DR

> **4 أوامر فرعية** تحت `forex alert`: `add`, `list`, `check`, `delete`. كل أمر يستدعي دالة `cmd_alert_*` في `commands.py`، مربوط بـ `set_defaults(func=...)` في `parser.py`. **النمط**: Sub-subparsers — نفس منطق Sprint 2 لكن بمستوى تعمق إضافي.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[forex alert] --> B{Sub-command}
    B -->|add| C[cmd_alert_add]
    B -->|list| D[cmd_alert_list]
    B -->|check| E[cmd_alert_check]
    B -->|delete| F[cmd_alert_delete]
    
    C --> G[AlertService.create_alert]
    D --> H[AlertService.list_alerts]
    E --> I[AlertService.check_alerts]
    F --> J[AlertService.delete_alert]

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style I fill:#bfb,stroke:#333,stroke-width:3px
```

---

## 🔬 التشريح — 4 دوال

### 1️⃣ `cmd_alert_add` — إنشاء

```mermaid
graph LR
    A[args: pair/condition/interval/message] --> B[AlertService]
    B --> C[create_alert]
    C --> D[print 5 fields]
    
    style C fill:#bfb,stroke:#333,stroke-width:2px
```

**المدخلات الإلزامية:** `--pair`, `--condition`. **الاختيارية:** `--interval` (`1day`), `--message`.

**المخرجات:** ID / Pair / Condition / Status / Message.

---

### 2️⃣ `cmd_alert_list` — عرض

```mermaid
graph TD
    A[--active flag] --> B[service.list_alerts]
    B --> C{empty?}
    C -->|Yes| D[print hint]
    C -->|No| E[print table]
    
    style E fill:#bfb,stroke:#333,stroke-width:2px
```

**جدول بـ 6 أعمدة:**

| العمود | المصدر |
|--------|--------|
| ID | `alert.id` |
| Pair | `alert.pair` |
| Timeframe | `alert.interval` |
| Condition | `metric operator target` |
| Status | 🟢 ACTIVE / ⚪ غير ذلك |
| Created At | `strftime` |

**🎯 تريكة ألوان emoji:** 🟢 للنشط، ⚪ لغيره — بديل خفيف لـ `rich`.

---

### 3️⃣ `cmd_alert_check` — **قلب الأوامر**

```mermaid
sequenceDiagram
    participant U as User
    participant C as cmd_alert_check
    participant S as AlertService
    participant F as ForexService

    U->>C: forex alert check
    C->>S: check_alerts(pair=None)
    S->>S: get active alerts
    S->>F: get/analyze for each
    F-->>S: current values
    S-->>C: list of triggered reports
    C->>U: print "N Alert(s) Triggered"
```

**حالتان:**

| الحالة | المخرجات |
|--------|---------|
| **لا شيء أُطلق** | `✅ All active alerts evaluated. No conditions triggered.` |
| **تم الإطلاق** | لكل واحد: `alert_id`, `pair`, `interval`, `condition`, `actual`, `message`, `triggered_at` |

**🎯 تريكة `--pair`:** فلترة اختيارية — `forex alert check --pair EUR/USD` يفحص زوجاً واحداً فقط.

---

### 4️⃣ `cmd_alert_delete` — حذف

| النتيجة | السلوك |
|---------|--------|
| `True` | `🗑️ Alert 'id' deleted successfully.` |
| `False` | `⚠️ Alert 'id' not found.` + `sys.exit(1)` |

**🎯 `exit(1)` عند الفشل:** يُمكّن السكربتات الخارجية من معرفة أن الحذف لم ينجح.

---

## 📊 ربط `parser.py` — Sub-subparsers

```mermaid
graph TD
    A[forex] --> B[alert]
    B --> C[add]
    B --> D[list]
    B --> E[check]
    B --> F[delete]
    
    C --> G["--pair, --condition, --interval, --message"]
    D --> H["--active"]
    E --> I["--pair"]
    F --> J["--id"]

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style G fill:#f9f,stroke:#333
    style J fill:#f9f,stroke:#333
```

**النمط:**
1. `subparsers.add_parser("alert")` — المستوى الأول
2. `p_alert.add_subparsers(dest="alert_command")` — **Sub-subparsers**
3. `alert_sub.add_parser("add")` — المستوى الثاني
4. `set_defaults(func=cmd_alert_add)` — **Dispatch** لكل مستوى

**🎯 `dest="alert_command"`:** يميز `alert` من `command` — مهم للـ debugging.

---

## 🧪 التحقق — 6 أوامر

| # | الأمر | المتوقع |
|---|-------|---------|
| 1 | `forex alert --help` | قائمة `add/list/check/delete` |
| 2 | `forex alert add --pair EUR/USD --condition "price > 1.05"` | Alert ACTIVE + ID |
| 3 | `forex alert add --pair EUR/USD --condition "rsi < 35"` | Alert ACTIVE |
| 4 | `forex alert list` | جدول بـ 2 تنبيهات |
| 5 | `forex alert check` | لا شيء (السعر لم يتجاوز) |
| 6 | `forex alert delete --id abc12345` | `🗑️ deleted` |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| `AlertService()` ينشئ `SQLiteStorage()` جديد | اتصال مكرر على DB نفسها |
| `cmd_alert_check` بدون `--limit` | يعتمد على `limit=35` الداخلي |
| `strftime("%Y-%m-%d %H:%M")` | يفقد الثواني في `list` |
| `cmd_alert_add` لا يعرض `interval` في الـ summary | مقصود |
| `exit(1)` عند `delete` غير موجود | صحيح — لكن قد يكسر pipelines |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Dispatch** | `set_defaults(func=...)` |
| **Sub-subparsers** | `alert` → `add/list/check/delete` |
| **Thin handlers** | كل `cmd_*` = 20-40 سطراً |
| **Service delegation** | كل المنطق في `AlertService` |
| **Fail Fast** | `required=True` لـ `--pair/--condition/--id` |
| **Fail Soft** | `list` فاضي → hint |
| **Exit codes** | `exit(1)` عند فشل الحذف |
| **Emoji UX** | 🟢/⚪/🔔/🗑️/⚠️ |

---

## 🗺️ Sprint 4

```mermaid
graph LR
    A["core/alert.py ✅"] --> B["AlertRepository ✅"]
    B --> C["AlertService ✅"]
    C --> D["CLI: alert commands 🆕"]
    D --> E["tests/test_alerts.py"]

    style D fill:#bfb,stroke:#333,stroke-width:3px
```


# 🧪 تحليل `test_alerts.py` — اختبارات Sprint 4

---

## 🎯 TL;DR

> **3 فئات اختبار** تغطي دورة حياة التنبيه كاملة: **Domain** (شروط) → **Storage** (CRUD) → **Service** (تقييم وإطلاق). تستخدم `:memory:` — معزولة وسريعة. الاختبار الأهم: **Idempotency** (لا إطلاق مكرر).

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[test_alerts.py] --> B[TestAlertConditionDomain]
    A --> C[TestAlertStorageAndCRUD]
    A --> D[TestAlertServiceEvaluation]
    
    B --> B1[parse valid]
    B --> B2[evaluate]
    B --> B3[invalid raises]
    
    C --> C1[save + retrieve]
    C --> C2[delete + non-existing]
    
    D --> D1[price trigger]
    D --> D2[idempotency]
    
    style D fill:#bfb,stroke:#333,stroke-width:2px
    style D2 fill:#f9f,stroke:#333,stroke-width:3px
```

---

## 🔬 التشريح — 3 فئات

### 1️⃣ `TestAlertConditionDomain` — اختبارات Domain

**لا تحتاج DB** — كائنات `AlertCondition` فقط.

| # | الاختبار | ما يتحقق منه |
|---|----------|---------------|
| 1 | `test_condition_parsing_valid` | `"price > 1.0850"` → metric/operator/value صحيحة |
| 2 | `test_condition_evaluation` | `≥ 1.0500` → True لـ `1.0500` و `1.0501`، False لـ `1.0499` و `None` |
| 3 | `test_invalid_condition_raises` | `"price ~ 1.05"` و `"price > abc"` → `ValidationError` |

**🎯 جوهر:** اختبار **الحدود** — `1.0500 == 1.0500` (مساواة) + `None` (فشل آمن).

---

### 2️⃣ `TestAlertStorageAndCRUD` — اختبارات DB

```mermaid
graph LR
    A[setUp: :memory:] --> B[test cases]
    B --> C[tearDown: disconnect]

    style A fill:#bbf,stroke:#333
    style C fill:#bbf,stroke:#333
```

| # | الاختبار | ما يتحقق منه |
|---|----------|---------------|
| 1 | `test_save_and_retrieve_alert` | `save_alert` → `get_alert_by_id` يعيد نفس البيانات |
| 2 | `test_delete_alert` | `delete` → `True` + `get` → `None` |
| 3 | (ضمني) | `delete("non_existing_id")` → `False` |

**🎯 تريكة `assert fetched is not None` بعد `assertIsNotNone`:** لأن `unittest` لا يفهم Type Narrowing — `mypy` يحتاج التأكيد لتجنب تحذيرات لاحقة.

**🎯 `setUp`/`tearDown`:** عزل كامل — كل اختبار له DB خاصة.

---

### 3️⃣ `TestAlertServiceEvaluation` — **الاختبار الأهم**

```mermaid
sequenceDiagram
    participant T as Test
    participant St as Storage
    participant S as AlertService

    T->>St: save_candle(close=1.0950)
    T->>S: create_alert("price >= 1.0950")
    T->>S: create_alert("price > 1.1000")
    T->>S: check_alerts(pair="EUR/USD")
    S->>St: get active alerts
    S->>St: get_history(limit=1)
    S->>S: evaluate
    S->>St: save_alert(TRIGGERED)
    S-->>T: [1 report]
    T->>S: check_alerts() AGAIN
    S-->>T: [] (لا تكرار!)
```

**الاختبارات المُتحقَّقة:**

| # | التحقق | النتيجة |
|---|--------|---------|
| 1 | عدد التنبيهات المُطلقة | 1 فقط |
| 2 | التنبيه الصحيح أُطلق | `alert_hit.id` |
| 3 | القيمة الفعلية | `1.0950` |
| 4 | الحالة في DB | `TRIGGERED` |
| 5 | `triggered_at` موجود | ✅ |
| 6 | التنبيه الآخر | ما زال `ACTIVE` |
| 7 | **إعادة الفحص** | **0 reports** (Idempotency) |

**🎯 جوهر الاختبار:** `second_check = check_alerts(...)` → `len == 0`. هذا يثبت أن `save_alert` بعد `trigger` **يمنع الإطلاق المتكرر**.

---

## 📊 تغطية الاختبار

```mermaid
pie title التغطية التقريبية
    "Domain (parse/evaluate)" : 30
    "Storage (CRUD)" : 25
    "Service (evaluation)" : 30
    "Idempotency" : 15
```

**التغطية الكلية:** ~75-85% من كود `alert.py`, `alert_service.py`, `sqlite_storage.py` (الجزء الخاص بالتنبيهات).

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| لا اختبار لـ `alert_list` أو `alert_delete` في CLI | مقصود — اختبار Domain/Service كافٍ |
| لا اختبار لـ `rsi < 35` trigger | يحسب RSI (يتطلب 14+ شمعة) — يُفضل إضافته |
| لا اختبار لـ `MemoryStorage` | `SQLiteStorage :memory:` يغطي نفس المنطق |
| `test_condition_evaluation` مختصر | يغطي `≥` فقط — يمكن إضافة باقي المعاملات |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Isolation** | `:memory:` لكل اختبار |
| **Layered Testing** | Domain → Storage → Service |
| **Edge Cases** | `None`, `==`, `<` |
| **Idempotency** | اختبار إعادة الفحص |
| **Assert Types** | `assertIsNotNone` + `assert None` |
| **Arrange-Act-Assert** | بنية واضحة في كل اختبار |

---

## 🗺️ Sprint 4

```mermaid
graph LR
    A["core/alert.py ✅"] --> B["AlertRepository ✅"]
    B --> C["AlertService ✅"]
    C --> D["CLI: alert commands ✅"]
    D --> E["tests 🆕"]
    E --> F["📄 docs + tag v0.4.0"]

    style E fill:#bfb,stroke:#333,stroke-width:3px
    style F fill:#ddd,stroke:#333
```

---

## 🧪 نتيجة التشغيل المتوقعة

```bash
python -m unittest discover tests
```

**المتوقع:**
```
...........................
----------------------------------------------------------------------
Ran 27 tests in 0.2xxs

OK
```

| Sprint | عدد الاختبارات |
|--------|----------------|
| 0 | 3 |
| 1 | ~5 |
| 2 | ~12 |
| 3 | ~4 |
| **4** | **3** |
| **المجموع** | **27** |

---

## 📋 الخطوة التالية

> **إغلاق Sprint 4**:
> 1. `docs/sprint_4_documentation.md` — توثيق الـ 5 خطوات.
> 2. تحديث `README.md` — إضافة `forex alert` + `Sprint 4` في الـ Roadmap.
> 3. تحديث `pyproject.toml` → `version = "0.4.0"`.
> 4. Commit + Tag:
>    ```bash
>    git add . && git commit -m "feat(alerts): complete Sprint 4 — Alert Engine"
>    git tag -a v0.4.0 -m "Sprint 4: Alerts & Triggers"
>    git push && git push --tags
>    ```

**🎯 معيار الإغلاق:** `27/27 tests OK` + `forex alert check` يعمل حياً + التوثيق جاهز.

**السؤال الجوهري قبل Sprint 5:** هل نُكمل إلى **Caching (Redis)**، أم نتوقف لمراجعة معمارية شاملة؟

**الجواب المقترح:** **مراجعة قصيرة** (15 دقيقة) قبل Sprint 5 — لأن Caching سيُغيّر `ForexService` جوهرياً، ويجب التأكد أن DI الحالي يدعمه بسهولة.